In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import xgboost as xgb
import sklearn
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, log_loss, confusion_matrix, classification_report


In [2]:
# Load the datasets
#elorating = pd.read_csv('C:/Users/mikko/OneDrive/python_projektit/ML_dec_tree_football/data_csv/EloRatings.csv')
matches = pd.read_csv('C:/Users/mikko/OneDrive/python_projektit/ML_dec_tree_football/data_csv/Matches.csv')

C:\Users\mikko\AppData\Local\Temp\ipykernel_12000\239085380.py:3: DtypeWarning: Columns (2) have mixed types. Specify dtype option on import or set low_memory=False.
  matches = pd.read_csv('C:/Users/mikko/OneDrive/python_projektit/ML_dec_tree_football/data_csv/Matches.csv')


In [3]:
matches_cleaned = matches.drop(columns=['MatchTime', 'HomeFouls', 'AwayFouls',
    'HomeYellow', 'AwayYellow', 'HomeRed', 'AwayRed',
    'OddHome', 'OddDraw', 'OddAway',
    'MaxHome', 'MaxDraw', 'MaxAway',
    'Over25', 'Under25', 'MaxOver25', 'MaxUnder25',
    'HandiSize', 'HandiHome', 'HandiAway',
    'C_LTH', 'C_LTA', 'C_VHD', 'C_VAD',
    'C_HTB', 'C_PHB'
])

In [4]:
matches_cleaned = matches_cleaned[matches_cleaned['Division'] == 'E0']
matches_cleaned['MatchDate'] = pd.to_datetime(matches_cleaned['MatchDate'])

matches_cleaned[['HomeTeam', 'AwayTeam']] = (
    matches_cleaned[['HomeTeam', 'AwayTeam']]
        .replace({
            "Nott'm Forest": "Nottingham Forest",
            "Nottm Forest": "Nottingham Forest"
        })
)

matches_cleaned['HomePoints'] = matches_cleaned['FTResult'].map({
    'H': 3,
    'D': 1,
    'A': 0
})

matches_cleaned['AwayPoints'] = matches_cleaned['FTResult'].map({
    'H': 0,
    'D': 1,
    'A': 3
})

In [5]:
# --- Prereqs: MatchDate datetime, Season (July->June), stable chronological order ---
matches_cleaned['MatchDate'] = pd.to_datetime(matches_cleaned['MatchDate'])

# Sort chronologically and create stable match id
matches_cleaned = matches_cleaned.sort_values('MatchDate').reset_index(drop=True)
matches_cleaned['MatchID'] = matches_cleaned.index

# --- Long format: one row per (match, team) with points earned ---
home_long = matches_cleaned[['MatchID', 'MatchDate', 'HomeTeam', 'HomePoints']].copy()
home_long = home_long.rename(columns={'HomeTeam': 'Team', 'HomePoints': 'Points'})
home_long['Side'] = 'Home'

away_long = matches_cleaned[['MatchID', 'MatchDate', 'AwayTeam', 'AwayPoints']].copy()
away_long = away_long.rename(columns={'AwayTeam': 'Team', 'AwayPoints': 'Points'})
away_long['Side'] = 'Away'

teams_long = pd.concat([home_long, away_long], ignore_index=True)

# --- Rolling average points over last 20 games per team (BEFORE match) ---
teams_long = teams_long.sort_values(['Team', 'MatchDate', 'MatchID']).reset_index(drop=True)

g = teams_long.groupby('Team')['Points']

# rolling avg of previous 20 matches: shift to exclude current match
teams_long['RollingAvgPoints20Before'] = (
    g.apply(lambda s: s.shift(1).rolling(window=20, min_periods=1).mean())
     .reset_index(level=0, drop=True)
)

# --- Merge back to match table as Home/Away features ---
home_feat = teams_long[teams_long['Side'] == 'Home'][['MatchID', 'RollingAvgPoints20Before']].rename(
    columns={'RollingAvgPoints20Before': 'HomeTotalPointsRoll20'}  # name kept, now rolling avg
)
away_feat = teams_long[teams_long['Side'] == 'Away'][['MatchID', 'RollingAvgPoints20Before']].rename(
    columns={'RollingAvgPoints20Before': 'AwayTotalPointsRoll20'}  # name kept, now rolling avg
)

matches_cleaned = matches_cleaned.merge(home_feat, on='MatchID', how='left')
matches_cleaned = matches_cleaned.merge(away_feat, on='MatchID', how='left')
# Optional cleanup:
# matches_cleaned = matches_cleaned.drop(columns=['MatchID'])

In [6]:
# Ensure chronological order
matches_cleaned['MatchDate'] = pd.to_datetime(matches_cleaned['MatchDate'])
matches_cleaned = matches_cleaned.sort_values('MatchDate').reset_index(drop=True)

# --- Rolling avg of last 20 HOME matches for each team ---
matches_cleaned['HomeTeamHomePointsRolling20'] = (
    matches_cleaned
    .groupby('HomeTeam')['HomePoints']
    .apply(lambda s: s.shift(1).rolling(window=20, min_periods=1).mean())
    .reset_index(level=0, drop=True)
)

In [7]:
matches_cleaned['AwayTeamAwayPointsRolling20'] = (
    matches_cleaned
    .groupby('AwayTeam')['AwayPoints']
    .apply(lambda s: s.shift(1).rolling(window=20, min_periods=1).mean())
    .reset_index(level=0, drop=True)
)

In [8]:
matches_cleaned.columns

Index(['Division', 'MatchDate', 'HomeTeam', 'AwayTeam', 'HomeElo', 'AwayElo',
       'Form3Home', 'Form5Home', 'Form3Away', 'Form5Away', 'FTHome', 'FTAway',
       'FTResult', 'HTHome', 'HTAway', 'HTResult', 'HomeShots', 'AwayShots',
       'HomeTarget', 'AwayTarget', 'HomeCorners', 'AwayCorners', 'HomePoints',
       'AwayPoints', 'MatchID', 'HomeTotalPointsRoll20',
       'AwayTotalPointsRoll20', 'HomeTeamHomePointsRolling20',
       'AwayTeamAwayPointsRolling20'],
      dtype='object')

In [9]:
# Ensure shots are numeric
matches_cleaned['HomeShots'] = pd.to_numeric(matches_cleaned['HomeShots'], errors='coerce')
matches_cleaned['AwayShots'] = pd.to_numeric(matches_cleaned['AwayShots'], errors='coerce')

# --- Long format: one row per (match, team) with shots taken in that match ---
home_long = matches_cleaned[['MatchID', 'MatchDate', 'HomeTeam', 'HomeShots']].copy()
home_long = home_long.rename(columns={'HomeTeam': 'Team', 'HomeShots': 'Shots'})
home_long['Side'] = 'Home'

away_long = matches_cleaned[['MatchID', 'MatchDate', 'AwayTeam', 'AwayShots']].copy()
away_long = away_long.rename(columns={'AwayTeam': 'Team', 'AwayShots': 'Shots'})
away_long['Side'] = 'Away'

teams_long = pd.concat([home_long, away_long], ignore_index=True)

# --- Rolling AVG shots per team over last 20 games (BEFORE match), no seasonality ---
teams_long = teams_long.sort_values(['Team', 'MatchDate', 'MatchID']).reset_index(drop=True)

g = teams_long.groupby('Team')['Shots']

teams_long['ShotsRolling20Before'] = (
    g.apply(lambda s: s.shift(1).rolling(window=20, min_periods=1).mean())
     .reset_index(level=0, drop=True)
)

# --- Merge back to match table as Home/Away features ---
home_feat = teams_long[teams_long['Side'] == 'Home'][['MatchID', 'ShotsRolling20Before']].rename(
    columns={'ShotsRolling20Before': 'HomeTeamShotsRolling20'}
)
away_feat = teams_long[teams_long['Side'] == 'Away'][['MatchID', 'ShotsRolling20Before']].rename(
    columns={'ShotsRolling20Before': 'AwayTeamShotsRolling20'}
)

matches_cleaned = matches_cleaned.merge(home_feat, on='MatchID', how='left')
matches_cleaned = matches_cleaned.merge(away_feat, on='MatchID', how='left')
# Optional cleanup:
# matches_cleaned = matches_cleaned.drop(columns=['MatchID'])

In [10]:
# Ensure datetime + chronological order
matches_cleaned['MatchDate'] = pd.to_datetime(matches_cleaned['MatchDate'])
matches_cleaned = matches_cleaned.sort_values('MatchDate').reset_index(drop=True)

# Shots numeric
matches_cleaned['HomeShots'] = pd.to_numeric(matches_cleaned['HomeShots'], errors='coerce')
matches_cleaned['AwayShots'] = pd.to_numeric(matches_cleaned['AwayShots'], errors='coerce')

# --- Home team: rolling avg of last 20 HOME matches (exclude current) ---
matches_cleaned['HomeTeamShotsHomeRolling20'] = (
    matches_cleaned
    .groupby('HomeTeam')['HomeShots']
    .apply(lambda s: s.shift(1).rolling(window=20, min_periods=1).mean())
    .reset_index(level=0, drop=True)
)

# --- Away team: rolling avg of last 20 AWAY matches (exclude current) ---
matches_cleaned['AwayTeamShotsAwayRolling20'] = (
    matches_cleaned
    .groupby('AwayTeam')['AwayShots']
    .apply(lambda s: s.shift(1).rolling(window=20, min_periods=1).mean())
    .reset_index(level=0, drop=True)
)

# Optional: fill early NaNs (teams with no prior home/away games)
# matches_cleaned[['HomeTeamShotsHomeRolling20', 'AwayTeamShotsAwayRolling20']] = (
#     matches_cleaned[['HomeTeamShotsHomeRolling20', 'AwayTeamShotsAwayRolling20']].fillna(0)
# )

In [11]:
# Ensure numeric
matches_cleaned['HomeTarget'] = pd.to_numeric(matches_cleaned['HomeTarget'], errors='coerce')
matches_cleaned['AwayTarget'] = pd.to_numeric(matches_cleaned['AwayTarget'], errors='coerce')

# Make sure sorted and MatchID exists
matches_cleaned['MatchDate'] = pd.to_datetime(matches_cleaned['MatchDate'])
matches_cleaned = matches_cleaned.sort_values('MatchDate').reset_index(drop=True)
if 'MatchID' not in matches_cleaned.columns:
    matches_cleaned['MatchID'] = matches_cleaned.index

# --- Long format ---
home_long = matches_cleaned[['MatchID','MatchDate','HomeTeam','HomeTarget']].copy()
home_long = home_long.rename(columns={'HomeTeam':'Team','HomeTarget':'TargetShots'})
home_long['Side'] = 'Home'

away_long = matches_cleaned[['MatchID','MatchDate','AwayTeam','AwayTarget']].copy()
away_long = away_long.rename(columns={'AwayTeam':'Team','AwayTarget':'TargetShots'})
away_long['Side'] = 'Away'

teams_long = pd.concat([home_long, away_long], ignore_index=True)

# --- Rolling AVG per team over last 20 matches (BEFORE match), no seasonality ---
teams_long = teams_long.sort_values(['Team','MatchDate','MatchID']).reset_index(drop=True)

g = teams_long.groupby('Team')['TargetShots']

teams_long['TargetShotsRolling20Before'] = (
    g.apply(lambda s: s.shift(1).rolling(window=20, min_periods=1).mean())
     .reset_index(level=0, drop=True)
)

# --- Merge back ---
home_feat = teams_long[teams_long['Side']=='Home'][['MatchID','TargetShotsRolling20Before']] \
    .rename(columns={'TargetShotsRolling20Before':'HomeTeamTargetShotsRolling20'})

away_feat = teams_long[teams_long['Side']=='Away'][['MatchID','TargetShotsRolling20Before']] \
    .rename(columns={'TargetShotsRolling20Before':'AwayTeamTargetShotsRolling20'})

matches_cleaned = matches_cleaned.merge(home_feat, on='MatchID', how='left')
matches_cleaned = matches_cleaned.merge(away_feat, on='MatchID', how='left')

In [12]:
# Ensure numeric
matches_cleaned['FTHome'] = pd.to_numeric(matches_cleaned['FTHome'], errors='coerce')
matches_cleaned['FTAway'] = pd.to_numeric(matches_cleaned['FTAway'], errors='coerce')

# --- Long format: goals scored by each team in each match ---
home_long = matches_cleaned[['MatchID', 'MatchDate', 'HomeTeam', 'FTHome']].copy()
home_long = home_long.rename(columns={'HomeTeam': 'Team', 'FTHome': 'Goals'})
home_long['Side'] = 'Home'

away_long = matches_cleaned[['MatchID', 'MatchDate', 'AwayTeam', 'FTAway']].copy()
away_long = away_long.rename(columns={'AwayTeam': 'Team', 'FTAway': 'Goals'})
away_long['Side'] = 'Away'

teams_long = pd.concat([home_long, away_long], ignore_index=True)

# --- Rolling AVG goals per team over last 20 matches (BEFORE match), no seasonality ---
teams_long = teams_long.sort_values(['Team', 'MatchDate', 'MatchID']).reset_index(drop=True)

g = teams_long.groupby('Team')['Goals']

teams_long['GoalsRolling20Before'] = (
    g.apply(lambda s: s.shift(1).rolling(window=20, min_periods=1).mean())
     .reset_index(level=0, drop=True)
)

# --- Merge back to match table ---
home_feat = teams_long[teams_long['Side'] == 'Home'][['MatchID', 'GoalsRolling20Before']].rename(
    columns={'GoalsRolling20Before': 'HomeTotalGoalsRolling20'}
)
away_feat = teams_long[teams_long['Side'] == 'Away'][['MatchID', 'GoalsRolling20Before']].rename(
    columns={'GoalsRolling20Before': 'AwayTotalGoalsRolling20'}
)

matches_cleaned = matches_cleaned.merge(home_feat, on='MatchID', how='left')
matches_cleaned = matches_cleaned.merge(away_feat, on='MatchID', how='left')

In [13]:
# Ensure numeric
matches_cleaned['FTHome'] = pd.to_numeric(matches_cleaned['FTHome'], errors='coerce')
matches_cleaned['FTAway'] = pd.to_numeric(matches_cleaned['FTAway'], errors='coerce')

# --- Home team: rolling avg goals in last 20 HOME matches (exclude current) ---
matches_cleaned['HomeGoalsHomeRolling20'] = (
    matches_cleaned
    .groupby('HomeTeam')['FTHome']
    .apply(lambda s: s.shift(1).rolling(window=20, min_periods=1).mean())
    .reset_index(level=0, drop=True)
)

# --- Away team: rolling avg goals in last 20 AWAY matches (exclude current) ---
matches_cleaned['AwayGoalsAwayRolling20'] = (
    matches_cleaned
    .groupby('AwayTeam')['FTAway']
    .apply(lambda s: s.shift(1).rolling(window=20, min_periods=1).mean())
    .reset_index(level=0, drop=True)
)

# Optional: fill early NaNs
# matches_cleaned[['HomeGoalsHomeRolling20','AwayGoalsAwayRolling20']] = (
#     matches_cleaned[['HomeGoalsHomeRolling20','AwayGoalsAwayRolling20']].fillna(0)
# )

In [14]:
# Ensure numeric
matches_cleaned['FTHome'] = pd.to_numeric(matches_cleaned['FTHome'], errors='coerce')
matches_cleaned['FTAway'] = pd.to_numeric(matches_cleaned['FTAway'], errors='coerce')

# --- Home team conceded at home: rolling avg conceded in last 20 HOME matches (exclude current) ---
matches_cleaned['HomeConcededHomeRolling20'] = (
    matches_cleaned
    .groupby('HomeTeam')['FTAway']
    .apply(lambda s: s.shift(1).rolling(window=20, min_periods=1).mean())
    .reset_index(level=0, drop=True)
)

# --- Away team conceded away: rolling avg conceded in last 20 AWAY matches (exclude current) ---
matches_cleaned['AwayConcededAwayRolling20'] = (
    matches_cleaned
    .groupby('AwayTeam')['FTHome']
    .apply(lambda s: s.shift(1).rolling(window=20, min_periods=1).mean())
    .reset_index(level=0, drop=True)
)

# Optional: fill early NaNs
# matches_cleaned[['HomeConcededHomeRolling20','AwayConcededAwayRolling20']] = (
#     matches_cleaned[['HomeConcededHomeRolling20','AwayConcededAwayRolling20']].fillna(0)
# )

In [15]:
mapping = {
    "H": 0,
    "D": 1,
    "A": 2
}

matches_cleaned["FTResult"] = matches_cleaned["FTResult"].map(mapping)

In [16]:
matches_cleaned.columns

Index(['Division', 'MatchDate', 'HomeTeam', 'AwayTeam', 'HomeElo', 'AwayElo',
       'Form3Home', 'Form5Home', 'Form3Away', 'Form5Away', 'FTHome', 'FTAway',
       'FTResult', 'HTHome', 'HTAway', 'HTResult', 'HomeShots', 'AwayShots',
       'HomeTarget', 'AwayTarget', 'HomeCorners', 'AwayCorners', 'HomePoints',
       'AwayPoints', 'MatchID', 'HomeTotalPointsRoll20',
       'AwayTotalPointsRoll20', 'HomeTeamHomePointsRolling20',
       'AwayTeamAwayPointsRolling20', 'HomeTeamShotsRolling20',
       'AwayTeamShotsRolling20', 'HomeTeamShotsHomeRolling20',
       'AwayTeamShotsAwayRolling20', 'HomeTeamTargetShotsRolling20',
       'AwayTeamTargetShotsRolling20', 'HomeTotalGoalsRolling20',
       'AwayTotalGoalsRolling20', 'HomeGoalsHomeRolling20',
       'AwayGoalsAwayRolling20', 'HomeConcededHomeRolling20',
       'AwayConcededAwayRolling20'],
      dtype='object')

In [17]:
matches_features = matches_cleaned[['HomeElo', 'AwayElo', 
    'Form3Home', 'Form5Home', 'Form3Away', 'Form5Away', 
    'FTResult', 
    'HomeTotalPointsRoll20', 'AwayTotalPointsRoll20',
    'HomeTeamHomePointsRolling20', 'AwayTeamAwayPointsRolling20', 
    'HomeTeamShotsRolling20', 'AwayTeamShotsRolling20', 
    'HomeTeamShotsHomeRolling20', 'AwayTeamShotsAwayRolling20', 
    'HomeTeamTargetShotsRolling20', 'AwayTeamTargetShotsRolling20', 
    'HomeTotalGoalsRolling20', 'AwayTotalGoalsRolling20', 
    'HomeGoalsHomeRolling20', 'AwayGoalsAwayRolling20', 
    'HomeConcededHomeRolling20','AwayConcededAwayRolling20']]

In [ ]:
matches_features['TotalPointsRoll20_diff'] = matches_features['AwayTotalPointsRoll20'] 
- matches_features['HomeTotalPointsRoll20']

matches_features['HomeAwayPointsRolling20_diff'] = matches_features['AwayTeamAwayPointsRolling20'] 
- matches_features['HomeTeamHomePointsRolling20']

matches_features['ShotsRolling20_diff'] = matches_features['AwayTeamShotsRolling20'] 
- matches_features['HomeTeamShotsRolling20']

matches_features['ShotsHomeAwayRolling20_diff'] = matches_features['AwayTeamShotsAwayRolling20'] 
- matches_features['HomeTeamShotsHomeRolling20']

matches_features['TargetShotsRolling20_diff'] = matches_features['AwayTeamTargetShotsRolling20'] 
- matches_features['HomeTeamTargetShotsRolling20']

matches_features['GoalsRolling20_diff'] = matches_features['AwayTotalGoalsRolling20'] 
- matches_features['HomeTotalGoalsRolling20']

matches_features['GoalsHomeAwayRolling20_diff'] = matches_features['AwayGoalsAwayRolling20'] 
- matches_features['HomeGoalsHomeRolling20']

matches_features['ConcededHomeAwayRolling20_diff'] = matches_features['AwayConcededAwayRolling20'] 
- matches_features['HomeConcededHomeRolling20']


In [19]:
matches_features

,HomeElo,AwayElo,Form3Home,Form5Home,Form3Away,Form5Away,FTResult,HomeTotalPointsRoll20,AwayTotalPointsRoll20,HomeTeamHomePointsRolling20,...,HomeConcededHomeRolling20,AwayConcededAwayRolling20,TotalPointsRoll20_diff,HomeAwayPointsRolling20_diff,ShotsRolling20_diff,ShotsHomeAwayRolling20_diff,TargetShotsRolling20_diff,GoalsRolling20_diff,GoalsHomeAwayRolling20_diff,ConcededHomeAwayRolling20_diff
0,1608.77,1579.99,0.0,0.0,0.0,0.0,0,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,1800.17,1681.36,0.0,0.0,0.0,0.0,0,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,1635.61,1679.18,0.0,0.0,0.0,0.0,2,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,1636.08,1630.02,0.0,0.0,0.0,0.0,1,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,1782.55,1685.55,0.0,0.0,0.0,0.0,0,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9405,1557.66,1985.45,1.0,2.0,4.0,8.0,2,0.30,1.90,0.35,...,2.55,0.90,1.90,1.90,14.15,11.90,4.55,1.65,1.80,0.90
9406,1882.33,1776.20,4.0,7.0,7.0,7.0,2,2.00,1.40,2.10,...,1.05,1.20,1.40,1.10,9.90,9.70,3.85,1.30,0.85,1.20
9407,1799.76,1880.42,5.0,8.0,6.0,12.0,2,1.70,1.55,1.60,...,1.00,1.40,1.55,1.55,16.05,13.80,5.55,1.30,1.65,1.40
9408,2008.67,1820.28,1.0,7.0,7.0,9.0,1,2.05,1.80,2.55,...,0.85,1.30,1.80,1.55,13.40,13.15,5.00,1.60,1.50,1.30


In [20]:
matches_features = matches_features.iloc[20:].reset_index(drop=True)

X = matches_features.drop('FTResult',axis=1)
y = matches_features['FTResult']

In [21]:
print(X.dtypes)

HomeElo                           float64
AwayElo                           float64
Form3Home                         float64
Form5Home                         float64
Form3Away                         float64
Form5Away                         float64
HomeTotalPointsRoll20             float64
AwayTotalPointsRoll20             float64
HomeTeamHomePointsRolling20       float64
AwayTeamAwayPointsRolling20       float64
HomeTeamShotsRolling20            float64
AwayTeamShotsRolling20            float64
HomeTeamShotsHomeRolling20        float64
AwayTeamShotsAwayRolling20        float64
HomeTeamTargetShotsRolling20      float64
AwayTeamTargetShotsRolling20      float64
HomeTotalGoalsRolling20           float64
AwayTotalGoalsRolling20           float64
HomeGoalsHomeRolling20            float64
AwayGoalsAwayRolling20            float64
HomeConcededHomeRolling20         float64
AwayConcededAwayRolling20         float64
TotalPointsRoll20_diff            float64
HomeAwayPointsRolling20_diff      

In [22]:
X = X.fillna(0)

In [23]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    shuffle=False
)

In [24]:
rf = RandomForestClassifier(
    n_estimators=1500,
    max_depth=None,
    min_samples_split=10,
    min_samples_leaf=5,
    max_features="sqrt",
    class_weight="balanced_subsample", 
    n_jobs=-1,
    random_state=42
)

rf.fit(X_train, y_train)

# Predict class + probabilities
pred_rf = rf.predict(X_test)
proba_rf = rf.predict_proba(X_test)

print("Accuracy:", accuracy_score(y_test, pred_rf))
print("LogLoss:", log_loss(y_test, proba_rf))
print("\nConfusion Matrix:\n", confusion_matrix(y_test, pred_rf))
print("\nClassification Report:\n", classification_report(y_test, pred_rf, digits=3))

Accuracy: 0.5106496272630457
LogLoss: 0.993361162257749

Confusion Matrix:
 [[555  82 177]
 [211  41 180]
 [193  76 363]]

Classification Report:
               precision    recall  f1-score   support

           0      0.579     0.682     0.626       814
           1      0.206     0.095     0.130       432
           2      0.504     0.574     0.537       632

    accuracy                          0.511      1878
   macro avg      0.430     0.450     0.431      1878
weighted avg      0.468     0.511     0.482      1878



In [25]:
imp = pd.Series(rf.feature_importances_, index=X_train.columns).sort_values(ascending=False)
print(imp.head(25))

HomeElo                           0.068846
AwayElo                           0.061226
HomeTeamShotsHomeRolling20        0.047003
HomeTeamShotsRolling20            0.044196
HomeGoalsHomeRolling20            0.040137
ShotsRolling20_diff               0.037985
HomeTeamTargetShotsRolling20      0.037938
AwayTeamShotsRolling20            0.037462
HomeTeamHomePointsRolling20       0.036361
HomeTotalPointsRoll20             0.036239
ShotsHomeAwayRolling20_diff       0.035259
AwayTeamShotsAwayRolling20        0.034917
HomeConcededHomeRolling20         0.033967
HomeTotalGoalsRolling20           0.033227
AwayTeamTargetShotsRolling20      0.031364
TargetShotsRolling20_diff         0.031164
TotalPointsRoll20_diff            0.030426
AwayTotalPointsRoll20             0.029486
HomeAwayPointsRolling20_diff      0.029438
AwayTeamAwayPointsRolling20       0.028620
GoalsRolling20_diff               0.027154
AwayTotalGoalsRolling20           0.026831
AwayConcededAwayRolling20         0.026283
ConcededHom

In [26]:
X_rf_last = X_test.iloc[[-1]]   # double brackets keep it 2D

pred_rf_last = rf.predict(X_rf_last)
proba_rf_last = rf.predict_proba(X_rf_last)

print("Predicted class:", pred_rf_last)
print("Probabilities:", proba_rf_last)

Predicted class: [2]
Probabilities: [[0.26451342 0.28039392 0.45509266]]


In [42]:
proba_rf_rng = rf.predict_proba(X_test)

rng = np.random.default_rng(42)

pred_rf_rng = np.array([
    rng.choice(len(p), p=p) for p in proba_rf_rng
])

In [43]:
print("Accuracy:", accuracy_score(y_test, pred_rf_rng))
print("LogLoss:", log_loss(y_test, proba_rf_rng))
print("\nConfusion Matrix:\n", confusion_matrix(y_test, pred_rf_rng))
print("\nClassification Report:\n", classification_report(y_test, pred_rf_rng, digits=3))

Accuracy: 0.4142705005324814
LogLoss: 0.993361162257749

Confusion Matrix:
 [[385 207 222]
 [151 128 153]
 [188 179 265]]

Classification Report:
               precision    recall  f1-score   support

           0      0.532     0.473     0.501       814
           1      0.249     0.296     0.271       432
           2      0.414     0.419     0.417       632

    accuracy                          0.414      1878
   macro avg      0.398     0.396     0.396      1878
weighted avg      0.427     0.414     0.419      1878



In [27]:
import optuna
from sklearn.model_selection import StratifiedKFold, cross_val_score

SEED = 42

def objective(trial):
    params = {
        "n_estimators": trial.suggest_int("n_estimators", 400, 2500),
        "max_depth": trial.suggest_int("max_depth", 4, 60),
        "min_samples_split": trial.suggest_int("min_samples_split", 2, 40),
        "min_samples_leaf": trial.suggest_int("min_samples_leaf", 1, 25),
        "max_features": trial.suggest_categorical("max_features", ["sqrt", "log2", None]),
        "bootstrap": trial.suggest_categorical("bootstrap", [True, False]),
        "class_weight": trial.suggest_categorical("class_weight", [None, "balanced", "balanced_subsample"]),
        "n_jobs": -1,
        "random_state": SEED,
    }

    model_rf = RandomForestClassifier(**params)

    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)

    # Optimize for probability quality (lower log loss is better)
    scores = cross_val_score(
        model_rf,
        X_train,
        y_train,
        cv=cv,
        scoring="neg_log_loss",
        n_jobs=-1
    )

    return float(np.mean(scores))  # maximize neg_log_loss

# Study
study = optuna.create_study(
    direction="maximize",
    sampler=optuna.samplers.TPESampler(seed=SEED)
)

study.optimize(objective, n_trials=60, show_progress_bar=True)

print("\nBest CV neg_log_loss:", study.best_value)
print("Best params:\n", study.best_params)

# Train final model with best params
rf_opt = RandomForestClassifier(
    **study.best_params,
    n_jobs=-1,
    random_state=SEED
)
rf_opt.fit(X_train, y_train)

c:\Users\mikko\AppData\Local\Programs\Python\Python313\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
[I 2026-03-05 18:08:54,147] A new study created in memory with name: no-name-3eb0a133-ea96-4ce7-98fc-155b0ab3819b
Best trial: 0. Best value: -0.99957:   2%|▏         | 1/60 [00:09<09:00,  9.16s/it]

[I 2026-03-05 18:09:03,302] Trial 0 finished with value: -0.9995697441394065 and parameters: {'n_estimators': 1186, 'max_depth': 58, 'min_samples_split': 30, 'min_samples_leaf': 15, 'max_features': 'sqrt', 'bootstrap': True, 'class_weight': 'balanced_subsample'}. Best is trial 0 with value: -0.9995697441394065.


Best trial: 0. Best value: -0.99957:   3%|▎         | 2/60 [00:28<14:38, 15.14s/it]

[I 2026-03-05 18:09:22,638] Trial 1 finished with value: -0.9997468976967921 and parameters: {'n_estimators': 2148, 'max_depth': 16, 'min_samples_split': 9, 'min_samples_leaf': 5, 'max_features': 'log2', 'bootstrap': False, 'class_weight': 'balanced_subsample'}. Best is trial 0 with value: -0.9995697441394065.


Best trial: 2. Best value: -0.999032:   5%|▌         | 3/60 [01:02<22:34, 23.76s/it]

...

[I 2026-03-05 18:26:00,145] Trial 57 finished with value: -0.9744261038741759 and parameters: {'n_estimators': 2468, 'max_depth': 8, 'min_samples_split': 27, 'min_samples_leaf': 21, 'max_features': 'sqrt', 'bootstrap': True, 'class_weight': None}. Best is trial 55 with value: -0.9739609697024789.


Best trial: 55. Best value: -0.973961:  98%|█████████▊| 59/60 [17:26<00:16, 16.04s/it]

[I 2026-03-05 18:26:20,421] Trial 58 finished with value: -1.005496851597281 and parameters: {'n_estimators': 2363, 'max_depth': 14, 'min_samples_split': 30, 'min_samples_leaf': 20, 'max_features': 'sqrt', 'bootstrap': False, 'class_weight': 'balanced'}. Best is trial 55 with value: -0.9739609697024789.


Best trial: 55. Best value: -0.973961: 100%|██████████| 60/60 [17:40<00:00, 17.68s/it]


[I 2026-03-05 18:26:35,089] Trial 59 finished with value: -0.9764757159411073 and parameters: {'n_estimators': 2234, 'max_depth': 24, 'min_samples_split': 33, 'min_samples_leaf': 16, 'max_features': 'sqrt', 'bootstrap': True, 'class_weight': None}. Best is trial 55 with value: -0.9739609697024789.

Best CV neg_log_loss: -0.9739609697024789
Best params:
 {'n_estimators': 2500, 'max_depth': 6, 'min_samples_split': 26, 'min_samples_leaf': 20, 'max_features': 'sqrt', 'bootstrap': True, 'class_weight': None}

TEST Accuracy: 0.5346112886048988
TEST LogLoss: 0.9840907024948748

Confusion Matrix:
 [[692   2 120]
 [295   0 137]
 [320   0 312]]

Classification Report:
               precision    recall  f1-score   support

           0      0.529     0.850     0.653       814
           1      0.000     0.000     0.000       432
           2      0.548     0.494     0.520       632

    accuracy                          0.535      1878
   macro avg      0.359     0.448     0.391      1878
weight

In [ ]:
# Evaluate on test set
pred_rf_opt = rf_opt.predict(X_test)
proba_rf_opt = rf_opt.predict_proba(X_test)

print("\nTEST Accuracy:", accuracy_score(y_test, pred_rf_opt))
print("TEST LogLoss:", log_loss(y_test, proba_rf_opt))
print("\nConfusion Matrix:\n", confusion_matrix(y_test, pred_rf_opt))
print("\nClassification Report:\n", classification_report(y_test, pred_rf_opt, digits=3))

In [28]:
proba_opt_rng = rf_opt.predict_proba(X_test)

rng = np.random.default_rng(42)

pred_opt_rng = np.array([
    rng.choice(len(p), p=p) for p in proba_opt_rng
])

In [29]:
print("TEST Accuracy:", accuracy_score(y_test, pred_opt_rng))
print("\nConfusion Matrix:\n", confusion_matrix(y_test, pred_rf))
print("\nClassification Report:\n", classification_report(y_test, pred_opt_rng, digits=3))

TEST Accuracy: 0.41533546325878595

Confusion Matrix:
 [[555  82 177]
 [211  41 180]
 [193  76 363]]

Classification Report:
               precision    recall  f1-score   support

           0      0.503     0.525     0.514       814
           1      0.243     0.250     0.246       432
           2      0.420     0.388     0.403       632

    accuracy                          0.415      1878
   macro avg      0.388     0.387     0.388      1878
weighted avg      0.415     0.415     0.415      1878



In [30]:
X_rf_last = X_test.iloc[[-1]]   # double brackets keep it 2D

pred_rf_last = rf_opt.predict(X_rf_last)
proba_rf_last = rf_opt.predict_proba(X_rf_last)

print("Predicted class:", pred_rf_last)
print("Probabilities:", proba_rf_last)

Predicted class: [2]
Probabilities: [[0.33420686 0.2787006  0.38709254]]


In [31]:
model_xb = xgb.XGBClassifier(
    objective="multi:softprob",
    num_class=3,
    n_estimators=5000,
    learning_rate=0.1,      
    max_depth=4,            
    min_child_weight=5,
    subsample=0.9,
    colsample_bytree=0.9,
    reg_lambda=1.0,
    tree_method="hist",
    eval_metric="mlogloss",
    early_stopping_rounds=200,  
    random_state=42,
)

model_xb.fit(X_train, y_train, eval_set=[(X_test, y_test)], verbose=100)

#metrics
proba_xb = model_xb.predict_proba(X_test)
pred_xb = proba_xb.argmax(axis=1)

print("Accuracy:", accuracy_score(y_test, pred_xb))
print("LogLoss:", log_loss(y_test, proba_xb))

print(confusion_matrix(y_test, pred_xb))
print(classification_report(y_test, pred_xb, digits=3))

[0]	validation_0-mlogloss:1.06025
[100]	validation_0-mlogloss:0.99690
[200]	validation_0-mlogloss:1.01355
[232]	validation_0-mlogloss:1.01830
Accuracy: 0.5319488817891374
LogLoss: 0.9848599713490491
[[683   3 128]
 [294   2 136]
 [314   4 314]]
              precision    recall  f1-score   support

           0      0.529     0.839     0.649       814
           1      0.222     0.005     0.009       432
           2      0.543     0.497     0.519       632

    accuracy                          0.532      1878
   macro avg      0.432     0.447     0.392      1878
weighted avg      0.463     0.532     0.458      1878



In [32]:
X_xb_w_last = X_test.iloc[[-1]]   # double brackets keep it 2D

pred_xb_last = model_xb.predict(X_xb_w_last)
proba_xb_last = model_xb.predict_proba(X_xb_w_last)

print("Predicted class:", pred_xb_last)
print("Probabilities:", proba_xb_last)

Predicted class: [2]
Probabilities: [[0.34750763 0.23340231 0.41909   ]]


In [ ]:
import numpy as np
import optuna
import xgboost as xgb

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score, log_loss, confusion_matrix, classification_report

SEED = 42
N_CLASSES = 3

# convert y if DataFrame with one column
if hasattr(y_train, "shape") and len(y_train.shape) == 2:
    y_train = y_train.iloc[:,0]
    y_test = y_test.iloc[:,0]

def objective(trial):

    params_xgbo = {
        "objective": "multi:softprob",
        "num_class": N_CLASSES,
        "tree_method": "hist",
        "eval_metric": "mlogloss",
        "random_state": SEED,
        "n_jobs": -1,

        # hyperparameters to tune
        "n_estimators": trial.suggest_int("n_estimators", 1000, 12000),
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.3, log=True),
        "max_depth": trial.suggest_int("max_depth", 2, 10),
        "min_child_weight": trial.suggest_float("min_child_weight", 1.0, 20.0),
        "subsample": trial.suggest_float("subsample", 0.5, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.5, 1.0),
        "reg_lambda": trial.suggest_float("reg_lambda", 1e-3, 50.0, log=True),

        "gamma": trial.suggest_float("gamma", 0.0, 10.0),
        "reg_alpha": trial.suggest_float("reg_alpha", 1e-8, 5.0, log=True),

        # early stopping goes HERE for xgboost >=3
        "early_stopping_rounds": 200,
    }

    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
    losses = []

    for tr_idx, va_idx in cv.split(X_train, y_train):

        X_tr = X_train.iloc[tr_idx]
        X_va = X_train.iloc[va_idx]
        y_tr = y_train.iloc[tr_idx]
        y_va = y_train.iloc[va_idx]

        model_xgb_o = xgb.XGBClassifier(**params_xgbo)

        model_xgb_o.fit(
            X_tr,
            y_tr,
            eval_set=[(X_va, y_va)],
            verbose=False
        )

        proba_xgbo = model_xgb_o.predict_proba(X_va)
        losses.append(log_loss(y_va, proba_xgbo, labels=np.arange(N_CLASSES)))

    return -np.mean(losses)


study_xgb = optuna.create_study(
    direction="maximize",
    sampler=optuna.samplers.TPESampler(seed=SEED)
)

study_xgb.optimize(objective, n_trials=60, show_progress_bar=True)

print("\nBest CV neg_log_loss:", study_xgb.best_value)
print("Best params:\n", study_xgb.best_params)


# ---- train final optimized model ----

best_params_xgb = {
    **study_xgb.best_params,
    "objective": "multi:softprob",
    "num_class": N_CLASSES,
    "tree_method": "hist",
    "eval_metric": "mlogloss",
    "random_state": SEED,
    "n_jobs": -1,
    "early_stopping_rounds": 200,
}

model_xb_opt = xgb.XGBClassifier(**best_params_xgb)

model_xb_opt.fit(
    X_train,
    y_train,
    eval_set=[(X_test, y_test)],
    verbose=100
)


In [34]:
# metrics
proba_xb_o = model_xb_opt.predict_proba(X_test)
pred_xb_o = proba_xb_o.argmax(axis=1)

print("\nAccuracy:", accuracy_score(y_test, pred_xb_o))
print("LogLoss:", log_loss(y_test, proba_xb_o))
print("\nConfusion Matrix:\n", confusion_matrix(y_test, pred_xb_o))
print("\nClassification Report:\n", classification_report(y_test, pred_xb_o, digits=3))


Accuracy: 0.5346112886048988
LogLoss: 0.9829567920245091

Confusion Matrix:
 [[667   1 146]
 [284   0 148]
 [293   2 337]]

Classification Report:
               precision    recall  f1-score   support

           0      0.536     0.819     0.648       814
           1      0.000     0.000     0.000       432
           2      0.534     0.533     0.534       632

    accuracy                          0.535      1878
   macro avg      0.357     0.451     0.394      1878
weighted avg      0.412     0.535     0.461      1878



In [35]:
proba_xgbo_opt_rng = model_xb_opt.predict_proba(X_test)

rng_xgbo = np.random.default_rng(42)

pred_xgbo_opt_rng = np.array([
    rng.choice(len(p), p=p) for p in proba_xgbo_opt_rng
])

In [36]:
print("TEST Accuracy:", accuracy_score(y_test, pred_xgbo_opt_rng))
print("\nConfusion Matrix:\n", confusion_matrix(y_test, pred_xgbo_opt_rng))
print("\nClassification Report:\n", classification_report(y_test, pred_xgbo_opt_rng, digits=3))

TEST Accuracy: 0.40628328008519704

Confusion Matrix:
 [[431 187 196]
 [200  98 134]
 [250 148 234]]

Classification Report:
               precision    recall  f1-score   support

           0      0.489     0.529     0.509       814
           1      0.226     0.227     0.227       432
           2      0.415     0.370     0.391       632

    accuracy                          0.406      1878
   macro avg      0.377     0.376     0.375      1878
weighted avg      0.404     0.406     0.404      1878



In [37]:
X_xb_o_last = X_test.iloc[[-1]]   # double brackets keep it 2D

pred_xb_o_last = model_xb_opt.predict(X_xb_o_last)
proba_xb_o_last = model_xb_opt.predict_proba(X_xb_o_last)

print("Predicted class:", pred_xb_o_last)
print("Probabilities:", proba_xb_o_last)

Predicted class: [2]
Probabilities: [[0.3205945 0.2548393 0.4245662]]
